# 01 — Pipeline Smoke Test

First end-to-end vertical slice of the CODEX CRC pipeline.

**Flow:** load TIFF → segment cells → extract per-cell marker intensities → save parquet → plot.

If `data/raw/` contains a `.tif` file, the notebook uses it directly. Otherwise it generates a synthetic CODEX-like image so the slice runs end-to-end without real data.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import tifffile

from codex_crc.features import extract_cell_features
from codex_crc.io import (
    load_codex_tiff,
    load_config,
    save_cells_parquet,
)
from codex_crc.plotting import plot_cells
from codex_crc.segmentation import run_mesmer, run_watershed

## 1. Load config

In [ ]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = load_config(REPO_ROOT / 'configs' / 'default.yaml')
config

## 2. Pick a TIFF (real if present, synthetic otherwise)

In [ ]:
raw_dir = REPO_ROOT / config['paths']['raw_dir']
raw_dir.mkdir(parents=True, exist_ok=True)
channel_names = config['channels']

real_tiffs = [p for p in raw_dir.glob('*.tif*') if not p.name.startswith('_synthetic')]

if real_tiffs:
    tiff_path = sorted(real_tiffs)[0]
    print(f'Using real data: {tiff_path.name}')
else:
    print('No real CODEX TIFF in data/raw/, generating synthetic demo image...')
    rng = np.random.default_rng(config['project']['random_seed'])
    n_ch = len(channel_names)
    H, W = 256, 256
    synthetic = rng.integers(0, 50, size=(n_ch, H, W), dtype=np.uint16)
    dapi_idx = channel_names.index('DAPI')
    membrane_idx = channel_names.index(config['segmentation']['membrane_channels'][0])
    yy, xx = np.ogrid[:H, :W]
    for _ in range(60):
        cy = int(rng.integers(15, H - 15))
        cx = int(rng.integers(15, W - 15))
        nuc_mask = (yy - cy) ** 2 + (xx - cx) ** 2 < 16
        mem_ring = ((yy - cy) ** 2 + (xx - cx) ** 2 < 36) & ~nuc_mask
        synthetic[dapi_idx][nuc_mask] = 800
        synthetic[membrane_idx][mem_ring] = 500
    tiff_path = raw_dir / '_synthetic_demo.tif'
    tifffile.imwrite(tiff_path, synthetic)
    print(f'Wrote synthetic image to {tiff_path}')

## 3. Load TIFF

In [ ]:
image, names = load_codex_tiff(tiff_path, channel_names=channel_names)
print(f'Image shape (C, H, W): {image.shape}')
print(f'Channels ({len(names)}): {names}')

## 4. Segment cells

In [ ]:
seg_cfg = config['segmentation']
backend = seg_cfg['backend']
print(f'Segmentation backend: {backend}')

if backend == 'mesmer':
    mask = run_mesmer(
        image,
        names,
        nuclear_channel=seg_cfg['nuclear_channel'],
        membrane_channels=seg_cfg['membrane_channels'],
        image_mpp=seg_cfg['image_mpp'],
    )
else:
    mask = run_watershed(image, names, seg_cfg['nuclear_channel'])

n_cells = int(len(set(np.unique(mask)) - {0}))
print(f'Segmented cells: {n_cells}')

## 5. Extract per-cell marker intensities

In [ ]:
cells = extract_cell_features(image, mask, names)
cells.head()

## 6. Save cell table to parquet

In [ ]:
processed_dir = REPO_ROOT / config['paths']['processed_dir']
out_path = processed_dir / f'cells_{tiff_path.stem}.parquet'
save_cells_parquet(cells, out_path)
print(f'Saved {len(cells)} cells to {out_path}')

## 7. Plot cells colored by DAPI

In [ ]:
results_dir = REPO_ROOT / config['paths']['results_dir']
fig = plot_cells(
    cells,
    color_by='DAPI',
    title=f'{tiff_path.stem}: cells colored by DAPI',
    save_path=results_dir / f'cells_{tiff_path.stem}_DAPI.png',
)
fig